<a href="https://colab.research.google.com/github/Joaoplims/NLP_Gametox/blob/main/NLP_Gametox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TODO


## 📋 Checklist de Execução do Experimento

### 🛠️ 1. Preparação dos Dados Reais (GameTox)

* [ ] **Carga e Filtragem:** Carregar o dataset do GameTox e filtrar as 42.963 mensagens pertencentes ao subconjunto em inglês.


* [ ] **Inspeção das Classes:** Mapear a contagem exata das 6 classes de *Intent* (*Non-Toxic*, *Insults and Flaming*, *Other Offensive Texts*, *Hate and Harassment*, *Threats*, *Extremism*) para documentar a linha de base do desbalanceamento.


* [ ] **Pré-processamento e Normalização:**
* [ ] Converter todo o texto para caixa baixa (*lowercase*).


* [ ] Tratar/remover caracteres especiais, pontuações desnecessárias.


* [ ] Criar uma rotina de tokenização adequada preservando pontuações e expressões típicas do contexto *gamer* (ex: *n00b*, *kys*, *fck*).


* [ ] **Garantia de Integridade:** Isolar o conjunto de **Teste Real** (nenhum dado sintético deve entrar aqui sob hipótese alguma).





---

### ⚙️ 2. Construção da Baseline (Sem Dados Sintéticos)

* [ ] **Vetorização dos Dados de Treino:**
* [ ] Extrair representações **TF-IDF** (testando uni-grams e bi-grams).
* [ ] Extrair/gerar representações densas via **Word2Vec** (treinado no próprio corpus de treino ou usando um modelo pré-treinado adaptado).


* [ ] **Treinamento de Modelos Clássicos:**
* [ ] Treinar os modelos selecionados (ex: *Regressão Logística*, *SVM*, *Naive Bayes*, *Random Forest*) apenas com o **Treino Real Desbalanceado**.




* [ ] **Avaliação e Métricas de Referência:**
* [ ] Avaliar a baseline no conjunto de **Teste Real**.


* [ ] Registrar o **Macro F1-Score**, o **F1-Score individual por classe** e a **Matriz de Confusão** (espera-se desempenho muito baixo ou nulo nas classes raras: *Threats*, *Extremism*, *Hate*).





---

### 🤖 3. Geração e Validação de Dados Sintéticos (LLM)

* [ ] **Engenharia de Prompt:**
* [ ] Escrever prompts estruturados (*Single-Agent* ou *Multi-Agent*) fornecendo exemplos reais (*few-shot*) das classes minoritárias (*Threats*, *Extremism*, *Hate and Harassment*).


* [ ] Instruir a LLM a simular a linguagem informal, ruidosa e cheia de gírias do ambiente de jogos digitais.




* [ ] **Geração e Filtragem:**
* [ ] Gerar amostras sintéticas suficientes para equilibrar proporcionalmente as classes raras em relação às classes majoritárias no conjunto de treino.


* [ ] Fazer uma checagem de qualidade/limpeza no texto gerado (remover saídas polidas demais ou respostas que fujam do formato do chat).




* [ ] **Consolidação das Bases de Treino:**
* [ ] **Treino A (Baseline):** 100% Real.


* [ ] **Treino B (Tradicional):** Real + SMOTE / Random Oversampling nos vetores.
* [ ] **Treino C (Proposta LLM):** Real + Dados Sintéticos gerados.





---

### 🧪 4. Experimentos Comparativos e Treinamento

* [ ] **Re-vetorizar** os novos conjuntos de treino estendidos (B e C) utilizando exatamente os mesmos parâmetros de TF-IDF e Word2Vec definidos no Passo 2.
* [ ] **Treinar os classificadores** em cada um dos cenários de treino (A, B e C) mantendo os mesmos hiperparâmetros.

---

### 📊 5. Avaliação Final, Análise e Discussão

* [ ] **Avaliação Padronizada:** Avaliar todos os modelos resultantes exatamente sobre o mesmo conjunto de **Teste Real** (mantido intacto).


* [ ] **Métricas e Tabelas:**
* [ ] Gerar a tabela comparativa contendo: **Precision**, **Recall**, **Macro F1-Score** e **F1-Score por classe** para cada modelo/cenário.


* [ ] Plotar as **Matrizes de Confusão** comparativas (Baseline vs. SMOTE vs. Sintético LLM).




* [ ] **Análise de Erros (Qualitativa):**
* [ ] Analisar falsos positivos e falsos negativos: O dado sintético ajudou o modelo a generalizar sem causar overfit ao estilo do LLM?


* [ ] Verificar se gírias neutras de jogos foram confundidas com frases tóxicas sintéticas.


# Dependencias:

In [5]:
pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=17639d918381a41e4101ae4230040fb062d43d144c80a405b7095a6aac2c0df0
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [12]:
pip install langid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941171 sha256=9a983904b3433a462766db1400ea92d792d189a0ae63d93b65401212a453d4b2
  Stored in directory: /root/.cache/pip/wheels/3c/bc/9d/266e27289b9019680d65d9b608c37bff1eff565b001c977ec5
Successfully built langid


# Datasets


In [7]:
val = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/val-gametox/val/val.csv"
train = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/train-gametox/train/train.csv"
test_label = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/test_release-gametox/test_release/test_index_label.csv"
test_msg = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/test_release-gametox/test_release/test_index_text.csv"

In [17]:
import pandas as pd
import re

# 1. Carregar os CSVs originais
df_train = pd.read_csv(train)
df_val = pd.read_csv(val)
df_test_msgs = pd.read_csv(test_msg)
df_test_lbls = pd.read_csv(test_label)
df_test = pd.merge(df_test_msgs, df_test_lbls, on='index')

# Função para checar se a mensagem NÃO contém caracteres cirílicos
def is_clean_english(text):
    return not bool(re.search(r'[\u0400-\u04FF]', str(text)))

# 2. Filtrar cada split
train_clean = df_train[df_train['message'].apply(is_clean_english)].copy()
val_clean   = df_val[df_val['message'].apply(is_clean_english)].copy()
test_clean  = df_test[df_test['message'].apply(is_clean_english)].copy()

# 3. Salvar os datasets filtrados
train_clean.to_csv('train_clean.csv', index=False)
val_clean.to_csv('val_clean.csv', index=False)
test_clean.to_csv('test_clean.csv', index=False)

print("=== SPLITS LIMPOS E PRONTOS ===")
print(f"Treino:     {len(train_clean)} amostras")
print(f"Validação:  {len(val_clean)} amostras")
print(f"Teste Real: {len(test_clean)} amostras")
print(f"Total:      {len(train_clean) + len(val_clean) + len(test_clean)} amostras")

=== SPLITS LIMPOS E PRONTOS ===
Treino:     39995 amostras
Validação:  4986 amostras
Teste Real: 4986 amostras
Total:      49967 amostras


In [18]:
import pandas as pd

# Mapeamento das classes conforme a documentação do GameTox
class_mapping = {
    0.0: "Non-Toxic",
    1.0: "Insults and Flaming",
    2.0: "Other Offensive Texts",
    3.0: "Hate and Harassment",
    4.0: "Threats",
    5.0: "Extremism"
}

# Contagem no conjunto de treino limpo
class_counts = train_clean['label'].value_counts().sort_index().rename(index=class_mapping)
class_percentages = (train_clean['label'].value_counts(normalize=True) * 100).sort_index().rename(index=class_mapping)

# Criando um DataFrame para visualização
imbalance_report = pd.DataFrame({
    'Amostras': class_counts,
    'Porcentagem (%)': class_percentages.map('{:.2f}%'.format)
})

print("=== DISTRIBUIÇÃO DAS CLASSES (TRAIN_CLEAN) ===")
print(imbalance_report)

=== DISTRIBUIÇÃO DAS CLASSES (TRAIN_CLEAN) ===
                       Amostras Porcentagem (%)
label                                          
Non-Toxic                 32511          81.29%
Insults and Flaming        5435          13.59%
Other Offensive Texts      1755           4.39%
Hate and Harassment         216           0.54%
Threats                      55           0.14%
Extremism                    23           0.06%


In [19]:
import re

def preprocess_gamer_text(text):
    if not isinstance(text, str):
        return ""
    # Converter para caixa baixa
    text = text.lower()
    # Remover caracteres especiais, mas manter o que é comum em chats (letras, números e espaços)
    # Mantemos alguns símbolos que podem ser usados em gírias ou censura (como *)
    text = re.sub(r'[^a-z0-9\s\*]', '', text)
    # Remover espaços extras
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Aplicando a limpeza nos DataFrames limpos
train_clean['message_preprocessed'] = train_clean['message'].apply(preprocess_gamer_text)
val_clean['message_preprocessed'] = val_clean['message'].apply(preprocess_gamer_text)
test_clean['message_preprocessed'] = test_clean['message'].apply(preprocess_gamer_text)

print("=== PRÉ-PROCESSAMENTO CONCLUÍDO ===")
print("Exemplos de antes e depois (Treino):")
display(train_clean[['message', 'message_preprocessed']].head(10))

=== PRÉ-PROCESSAMENTO CONCLUÍDO ===
Exemplos de antes e depois (Treino):


,message,message_preprocessed
0,no rush,no rush
1,whatever ... watch the replay,whatever watch the replay
2,useless,useless
3,3 gunmark,3 gunmark
4,lol,lol
5,i softened him lol,i softened him lol
6,come,come
7,stupid kids grow up,stupid kids grow up
8,fking light diference,fking light diference
9,"hori, info ?",hori info
